# Dataset Validation

This notebook runs a set of independent checks against the two source
datasets produced by the Stanford and govinfo notebooks, reading each
saved file directly from disk rather than relying on any counter or
message produced during collection. It is intended to be the final
check before either dataset is promoted (manually) to the shared Drive
folder for use by collaborators.

## Structure of this notebook

1. Connect to Google Drive
2. Stanford dataset checks
3. govinfo dataset checks
4. Cross-dataset compatibility notes, relevant for a future merge step

## Section 1: Connect to Google Drive

Both datasets are read from the shared working folder that the
Stanford and govinfo notebooks save to.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

WORKING_FOLDER = "/content/drive/MyDrive/Work Project 2026: Datasets (raw)"

import os
print("Files in working folder:")
for name in os.listdir(WORKING_FOLDER):
    print(" -", name)

Mounted at /content/drive
Files in working folder:
 - verify_mount_test.txt
 - congress_speeches_2016_present.jsonl


## Section 2: Stanford dataset checks

Checks: row count, date range, per-session coverage, party value
distribution, and duplicate speech IDs.

In [2]:
import pandas as pd

stanford_path = f"{WORKING_FOLDER}/congress_speeches_2001_2017.parquet"
stanford_df = pd.read_parquet(stanford_path)

print("=== BASIC SHAPE ===")
print(f"Rows: {stanford_df.shape[0]}")
print(f"Columns: {stanford_df.columns.tolist()}")

=== BASIC SHAPE ===
Rows: 823341
Columns: ['speech_id', 'speech', 'chamber', 'date', 'number_within_file', 'speaker', 'first_name', 'last_name', 'state', 'gender', 'line_start', 'line_end', 'file', 'char_count', 'word_count', 'speakerid', 'party', 'state_map', 'chamber_map', 'speaker_first_name', 'congress', 'date_parsed', 'icpsr', 'party_crosscheck']


In [3]:
print("=== CONGRESS SESSION COVERAGE ===")
print(stanford_df["congress"].value_counts().sort_index())

print("\n=== PARTY DISTRIBUTION ===")
print(stanford_df["party"].value_counts(dropna=False))

print("\n=== DATE RANGE ===")
stanford_df["date_parsed"] = pd.to_datetime(stanford_df["date"], format="%Y%m%d", errors="coerce")
print(f"Earliest speech: {stanford_df['date_parsed'].min()}")
print(f"Latest speech:   {stanford_df['date_parsed'].max()}")

print("\n=== DUPLICATE CHECK ===")
print(f"Duplicate speech_id entries: {stanford_df['speech_id'].duplicated().sum()}")

print("\n=== MISSING TEXT CHECK ===")
print(f"Speeches with no text at all: {stanford_df['speech'].isna().sum()}")

print("\n=== CHAMBER SHARE PER SESSION ===")
chamber_by_session = stanford_df.groupby(["congress", "chamber"]).size().unstack(fill_value=0)
chamber_by_session["senate_share"] = chamber_by_session.get("S", 0) / chamber_by_session.sum(axis=1)
print(chamber_by_session)

=== CONGRESS SESSION COVERAGE ===
congress
107    116866
108    127963
109    117130
110    130090
111     94641
112     91060
113     77621
114     67970
Name: count, dtype: int64

=== PARTY DISTRIBUTION ===
party
D    441536
R    377021
I      4730
P        37
A        17
Name: count, dtype: int64

=== DATE RANGE ===
Earliest speech: 2001-01-03 00:00:00
Latest speech:   2016-09-09 00:00:00

=== DUPLICATE CHECK ===
Duplicate speech_id entries: 0

=== MISSING TEXT CHECK ===
Speeches with no text at all: 0

=== CHAMBER SHARE PER SESSION ===
chamber       H      S  senate_share
congress                            
107       54546  62305      0.533200
108       62134  65816      0.514388
109       63645  53470      0.456560
110       78132  51956      0.399391
111       56864  37743      0.398945
112       58124  32924      0.361611
113       47894  29721      0.382929
114       43692  24278      0.357187


In [4]:
print("=== SAMPLE ROWS ===")
print(stanford_df[["date", "congress", "chamber", "speaker", "party", "speech"]].sample(5, random_state=1))

=== SAMPLE ROWS ===
            date  congress chamber               speaker party  \
273404  20050623       109       S             Mr. GREGG     R   
725002  20140227       113       H        Mr. HENSARLING     R   
108126  20020926       107       S           Mr. BENNETT     R   
171101  20030924       108       H            Mr. DUNCAN     R   
174908  20031015       108       H  Mr. KING of New York     R   

                                                   speech  
273404              Will the Senator yield on that point?  
725002   Mr. Speaker. I rise in opposition to the motion.  
108126  I am happy to yield. with the understanding I ...  
171101  Mr. Chairman. I yield myself such time as I ma...  
174908  Mr. Speaker. I urge passage of the Senate bill...  


In [5]:
print("=== ICPSR COVERAGE AND PARTY CROSSCHECK ===")
icpsr_coverage = stanford_df["icpsr"].notna().mean()
print(f"ICPSR resolved for {icpsr_coverage:.1%} of speeches")

party_code_map = {"Democrat": "D", "Republican": "R", "Independent": "I"}
both_present = stanford_df[stanford_df["party_crosscheck"].notna()].copy()
both_present["party_crosscheck_mapped"] = both_present["party_crosscheck"].map(party_code_map)
agreement = (both_present["party"] == both_present["party_crosscheck_mapped"]).mean()
print(f"Rows with both party values available: {len(both_present)}")
print(f"Agreement between Stanford's party and the ICPSR crosscheck: {agreement:.1%}")

=== ICPSR COVERAGE AND PARTY CROSSCHECK ===
ICPSR resolved for 97.7% of speeches
Rows with both party values available: 810632
Agreement between Stanford's party and the ICPSR crosscheck: 100.0%


### Interpreting the Stanford results

As a concrete reference, a confirmed run produced: 823,341 rows across
sessions 107-114, 0 duplicate speech IDs, and a date range from
2001-01-03 to 2016-09-09 (see Section 4 of the Stanford notebook for
the confirmed end date and its implications for the govinfo dataset's
gap-filling period).

Two details found in a confirmed run are worth noting for anyone
extending this dataset:

- **The full column list is larger than a minimal description would
  suggest** - alongside the documented columns, the raw session files
  also carry `number_within_file`, `last_name`, `line_start`,
  `line_end`, `file`, `char_count`, `state_map`, and `chamber_map`.
  `speaker_first_name` is also present - unlike the others, this one is
  functionally used (Section 5 of the Stanford notebook, for ICPSR
  matching), not just an inert extra column.
- **Party values are not limited to D/R/I.** A confirmed run showed:
  D (441,536), R (377,021), I (4,730), P (37), A (17). The rarer codes
  (P, A) correspond to minor historical parties; any downstream
  analysis that assumes only three party values should account for
  this.

**ICPSR coverage and party crosscheck**, confirmed run: ICPSR resolved
for 97.7% of speeches - notably higher than the govinfo dataset's
75.8%, consistent with Stanford already providing separated first/last
name and state fields, which the govinfo dataset has to first extract
from raw text. Among the 810,632 rows where both Stanford's own `party`
and the independently-derived `party_crosscheck` are available, the
two agree in all but 90 rows (99.99%). Investigating these 90 rows
found they concentrate almost entirely in two individuals, rather than
being spread across many different speakers:

- **53 rows: Acevedo-Vilá**, Puerto Rico's non-voting Resident
  Commissioner (2001-2005). Non-voting territorial delegates can have
  complex or non-standard party representations in the reference data
  (here, affiliated with Puerto Rico's local Popular Democratic Party
  while caucusing with U.S. Democrats in Congress); one specific term
  entry in the reference data shows an anomalous value in the party
  field that appears to be a state abbreviation rather than a party
  name, likely a data quality issue in the upstream reference source
  itself rather than in this notebook's own logic.
- **34 rows: Crenshaw**, a Representative Stanford's own `party` column
  records as "I" (Independent) throughout, despite him being a
  Republican for his entire tenure - this looks like a genuine error
  in Stanford's own party coding, correctly surfaced by the crosscheck
  rather than caused by it.
- The remaining few rows (a "Lee" from New York, 3 occurrences) were
  not further investigated given how small a share of the total they
  represent.

None of this points to a systematic problem with the matching logic
itself: the disagreements are concentrated in a small number of
individuals with known, explainable causes (a non-voting delegate's
complex party affiliation, and an apparent pre-existing error in
Stanford's own data), not spread broadly across the dataset.

## Section 3: govinfo dataset checks

Checks: row count, per-year coverage, exact duplicate records,
party-match rate and the most frequent unmatched names, speech length
distribution, and a manual read-through of a random sample of entries.

In [6]:
import json
from collections import Counter

govinfo_path = f"{WORKING_FOLDER}/congress_speeches_2016_present.jsonl"

govinfo_records = []
skipped_lines = 0
with open(govinfo_path) as f:
    for line in f:
        line = line.strip()
        if not line:
            skipped_lines += 1
            continue
        try:
            govinfo_records.append(json.loads(line))
        except json.JSONDecodeError:
            skipped_lines += 1

print(f"Total records loaded: {len(govinfo_records)}")
if skipped_lines:
    print(f"Skipped {skipped_lines} blank/invalid line(s).")

Total records loaded: 276513


In [7]:
print("=== SPEECHES PER YEAR ===")
year_counts = Counter(r["date"][:4] for r in govinfo_records)
for year in sorted(year_counts):
    print(f"  {year}: {year_counts[year]:,}")

print("\n=== CHAMBER SHARE PER YEAR ===")
# Directly checks the previously flagged issue: an earlier version of
# the govinfo notebook silently dropped Senate speeches on many days
# (a document-listing pagination bug - see 02_govinfo_dataset.ipynb,
# Section 4), reducing the Senate's share of collected speeches by
# roughly 70% relative to Stanford's equivalent years. Compare this
# against Stanford's per-session chamber share (Section 2) - the two
# should be broadly comparable in years close to the source break,
# rather than showing a large, sudden drop in Senate share.
import pandas as pd
chamber_year_counts = Counter((r["date"][:4], r["chamber"]) for r in govinfo_records)
years = sorted(set(y for y, c in chamber_year_counts))
chamber_df = pd.DataFrame(
    {"HOUSE": [chamber_year_counts.get((y, "HOUSE"), 0) for y in years],
     "SENATE": [chamber_year_counts.get((y, "SENATE"), 0) for y in years]},
    index=years,
)
chamber_df["senate_share"] = chamber_df["SENATE"] / (chamber_df["HOUSE"] + chamber_df["SENATE"])
print(chamber_df)

=== SPEECHES PER YEAR ===
  2016: 6,982
  2017: 36,367
  2018: 28,632
  2019: 33,399
  2020: 24,559
  2021: 27,047
  2022: 24,021
  2023: 26,103
  2024: 24,546
  2025: 26,414
  2026: 18,443

=== CHAMBER SHARE PER YEAR ===
      HOUSE  SENATE  senate_share
2016   4527    2455      0.351618
2017  24808   11559      0.317843
2018  18728    9904      0.345907
2019  23598    9801      0.293452
2020  14688    9871      0.401930
2021  16482   10565      0.390616
2022  15216    8805      0.366554
2023  18823    7280      0.278895
2024  17606    6940      0.282734
2025  15580   10834      0.410161
2026  12215    6228      0.337689


In [8]:
print("=== DUPLICATE CHECK ===")
seen = set()
duplicates = 0
for r in govinfo_records:
    key = (r["date"], r["speaker"], r["speech"])
    if key in seen:
        duplicates += 1
    seen.add(key)
print(f"Exact duplicate records found: {duplicates}")
print("(Should be 0 if the govinfo notebook's deduplication step already ran.)")

=== DUPLICATE CHECK ===
Exact duplicate records found: 0
(Should be 0 if the govinfo notebook's deduplication step already ran.)


In [9]:
print("=== PARTY-MATCH RATE ===")
matched = sum(1 for r in govinfo_records if r["party"] is not None)
total = len(govinfo_records)
print(f"Total speeches: {total:,}")
print(f"Matched to a party: {matched:,} ({100*matched/total:.1f}%)")

unmatched_names = Counter(r["speaker"] for r in govinfo_records if r["party"] is None)
print("\nTop 20 most frequent unmatched names:")
for name, count in unmatched_names.most_common(20):
    print(f"  {count:5d}x  {name}")

=== PARTY-MATCH RATE ===
Total speeches: 276,513
Matched to a party: 266,803 (96.5%)

Top 20 most frequent unmatched names:
   1181x  LEE
    936x  BROWN
    875x  COLLINS
    756x  MURPHY
    648x  PETERS
    646x  REED
    609x  KENNEDY
    508x  RODGERS
    380x  MORAN
    318x  SMITH
    268x  JOHNSON
    210x  HARRIS
    208x  YOUNG
    167x  KELLY
    164x  MENENDEZ
    141x  EDDIE BERNICE JOHNSON
    134x  KIM
    124x  KING
    121x  PEREZ
     97x  GOLDMAN


In [10]:
print("=== ICPSR COVERAGE ===")
has_icpsr = sum(1 for r in govinfo_records if r.get("icpsr") is not None)
print(f"Records with an ICPSR id: {has_icpsr:,} ({100*has_icpsr/total:.1f}%)")
print("(Expected to closely track the party-match rate above, since both",
      "come from the same name-resolution step.)")

=== ICPSR COVERAGE ===
Records with an ICPSR id: 238,919 (86.4%)
(Expected to closely track the party-match rate above, since both come from the same name-resolution step.)


In [11]:
print("=== STATE FIELD COVERAGE ===")
# The state named alongside a speaker (e.g. "of Texas") is optional in
# the source text and used only as an additional disambiguating detail
# (see 02_govinfo_dataset.ipynb, Sections 2-3) - a low coverage number
# here is expected and not a sign of a problem on its own, since most
# speeches do not need a state to be resolved unambiguously.
has_state = sum(1 for r in govinfo_records if r.get("state"))
print(f"Records with a captured state: {has_state:,} ({100*has_state/total:.1f}%)")

# Among records that were NOT matched to a party, how many at least had
# a captured state that, for some other reason (e.g. an unusual name
# format), still didn't lead to a unique match? A high count here would
# suggest the state tie-breaker itself needs further investigation.
unmatched_with_state = sum(1 for r in govinfo_records if r["party"] is None and r.get("state"))
unmatched_total = sum(1 for r in govinfo_records if r["party"] is None)
if unmatched_total:
    print(f"Of {unmatched_total:,} unmatched records, {unmatched_with_state:,} "
          f"({100*unmatched_with_state/unmatched_total:.1f}%) had a captured state "
          f"but were still not uniquely resolved.")

=== STATE FIELD COVERAGE ===
Records with a captured state: 40,642 (14.7%)
Of 9,710 unmatched records, 763 (7.9%) had a captured state but were still not uniquely resolved.


In [12]:
print("=== SPEECH LENGTH DISTRIBUTION ===")
import statistics

word_counts = [len(r["speech"].split()) for r in govinfo_records]
print(f"Median words per speech: {statistics.median(word_counts)}")
print(f"Mean words per speech: {statistics.mean(word_counts):.1f}")
print(f"Min: {min(word_counts)}, Max: {max(word_counts)}")

sorted_by_length = sorted(govinfo_records, key=lambda r: len(r["speech"].split()), reverse=True)
print("\nTop 10 longest speeches:")
for r in sorted_by_length[:10]:
    wc = len(r["speech"].split())
    print(f"  {wc:7d} words  -  {r['speaker']:20}  {r['date']}")

=== SPEECH LENGTH DISTRIBUTION ===
Median words per speech: 171
Mean words per speech: 395.6
Min: 4, Max: 29910

Top 10 longest speeches:
    29910 words  -  ROYCE                 2017-07-25
    29890 words  -  HENSARLING            2018-05-22
    29847 words  -  DIAZ-BALART           2023-09-29
    29784 words  -  McCONNELL             2018-03-08
    29525 words  -  JACKSON LEE           2018-06-26
    29513 words  -  FRELINGHUYSEN         2017-03-08
    29328 words  -  DeGETTE               2020-09-24
    29093 words  -  McCOLLUM              2023-03-29
    29002 words  -  CAROLYN B. MALONEY    2021-12-09
    28910 words  -  HILL                  2026-02-09


In [13]:
print("=== RANDOM SAMPLE (manual read-through) ===")
import random

random.seed(42)
sample = random.sample(govinfo_records, 5)
for r in sample:
    print(f"--- {r['speaker']} ({r['party']}) - {r['date']} - {r['chamber']} ---")
    print(r["speech"][:250])
    print("...")
    print()

=== RANDOM SAMPLE (manual read-through) ===
--- DOGGETT (Democrat) - 2018-04-12 - HOUSE ---
Mr. Speaker, like some stormy sermon from Trump on the 
virtues of chastity, I believe these House Republicans today really do 
deserve a gold medal for hypocrisy.
  After approving their budget-busting, trickle-down, trillion-dollar 
tax break and r
...

--- SMITH (Democrat) - 2016-12-02 - HOUSE ---
Mr. Speaker, I yield 2\1/2\ minutes to the 
gentlewoman from Guam (Ms. Bordallo), the ranking member on the 
Subcommittee on Readiness.
  (Ms. BORDALLO asked and was given permission to revise and extend her 
remarks.)
...

--- LESKO (Republican) - 2021-06-22 - HOUSE ---
Mr. Speaker, I rise today to honor the important 
contributions of amateur radio operators in Arizona and across the 
Nation.
  Throughout our Nation's history, amateur radio has served an 
important purpose. From providing a means of communication i
...

--- ROUZER (Republican) - 2020-11-17 - HOUSE ---
Mr. Speaker, I yield myself s

### Interpreting the govinfo results

As a concrete reference, a confirmed run of the complete dataset
(September 10, 2016 through the most recent partial year) produced:
276,513 total records, 0 exact duplicates, an 96.5% party-match rate,
and an 86.4% ICPSR-match rate. The longest speech observed was 29,910
words, exactly at the `MAX_SPEECH_WORDS` cap, as expected (the cap
excludes anything longer, so a value at the boundary is normal, not a
sign of a problem).

**The Senate undercount is confirmed resolved.** The chamber-share
check above is the direct test of the pagination fix described in
`02_govinfo_dataset.ipynb`, Section 4: the Senate's share of speeches
now ranges from 27.9% to 41.0% across years, consistent with Stanford's
own declining trend over its last sessions, rather than the
11.5%-30.8% range seen before the fix. The two datasets meet almost
exactly at the handover point: Stanford's last session (114,
2015-2016) shows a 35.7% Senate share; govinfo's first year (2016)
shows 35.2% - a close match where a sharp discontinuity would
otherwise have suggested a remaining problem.

**ICPSR coverage (86.4%) is close to, but still somewhat below, the
party-match rate (96.5%).** The explanation already documented for
the pre-fix dataset (non-voting territorial delegates structurally
never receive an ICPSR id, regardless of how reliably their name
resolves to a party) continues to apply.

**The state field (Sections 2-3) resolved a meaningful share of
previously-ambiguous cases.** Only 14.7% of all records had a captured
state at all (most speaker names do not need one for disambiguation),
but among records that remained unmatched, only 7.9% of those that did
have a captured state still failed to resolve - meaning the state
tie-breaker succeeds in the large majority of cases where it has
something to work with.

**A minor, known data-quality artifact**: one sample record (Jordan,
2016-09-14) shows trailing HTML markup (`</pre></body></html>`)
appended to the end of a speech, apparently left over at a document
boundary that the speech-extraction pattern did not fully clean up.
This was not addressed, since it appears limited to a small amount of
trailing boilerplate at the end of a document's final speech rather
than affecting the speech's substantive content, and does not warrant
the cost of a further fix at this time.

## Section 4: Cross-dataset compatibility notes

The following differences between the two datasets' raw column
formats are relevant to a future step that combines them into one
unified dataset, and are noted here rather than resolved:

- **Party values differ in format**: Stanford uses single-letter codes
  (`D`, `R`, `I`); govinfo uses full words (`Democrat`, `Republican`).
  A merge step needs to map one representation to the other.
- **Date format differs**: Stanford stores `YYYYMMDD` as a string;
  govinfo stores `YYYY-MM-DD`.
- **ICPSR coverage differs**: govinfo records include an `icpsr` field
  where a match was found; the Stanford dataset does not include ICPSR
  identifiers at all (not present in the source release).
- **Time range boundary**: the Stanford dataset's actual latest date
  and the govinfo dataset's earliest date (2017-01-01) should be
  checked against each other for any gap or overlap once both
  datasets' real date ranges are confirmed (see Section 4 of the
  Stanford notebook).

## Section 5: Gap-filling period checks (2016-09-10 to 2016-12-31)

This period was collected using a pipeline validated only against
2017 and later. Two representative weeks within this range are
checked separately here, at the same level of detail as the
diverse-week validation in the govinfo notebook, before this period
is trusted alongside the rest of the dataset.

In [14]:
# Two representative weeks within the gap-filling period: an ordinary
# session week, and a week covering the November 2016 election
# aftermath (a period of likely high floor-speech activity).
import statistics

gap_weeks = [
    ("2016-09-12", "2016-09-18"),
    ("2016-11-14", "2016-11-20"),
]

for start, end in gap_weeks:
    week_records = [r for r in govinfo_records if start <= r["date"] <= end]
    matched = sum(1 for r in week_records if r["party"] is not None)
    total = len(week_records)
    print(f"=== {start} to {end} ===")
    print(f"Total speeches: {total}")
    if total > 0:
        print(f"Matched to a party: {matched} ({100*matched/total:.1f}%)")
        word_counts = [len(r["speech"].split()) for r in week_records]
        print(f"Median words: {statistics.median(word_counts)}, Max: {max(word_counts)}")
    print()

=== 2016-09-12 to 2016-09-18 ===
Total speeches: 1273
Matched to a party: 1248 (98.0%)
Median words: 162, Max: 13252

=== 2016-11-14 to 2016-11-20 ===
Total speeches: 668
Matched to a party: 661 (99.0%)
Median words: 186.5, Max: 7675



In [15]:
# Manual read-through of a few entries from this specific period.
import random

gap_period_records = [r for r in govinfo_records if "2016-09-10" <= r["date"] <= "2016-12-31"]
random.seed(42)
sample = random.sample(gap_period_records, min(5, len(gap_period_records)))
for r in sample:
    print(f"--- {r['speaker']} ({r['party']}) - {r['date']} - {r['chamber']} ---")
    print(r["speech"][:250])
    print("...")
    print()

--- THORNBERRY (Republican) - 2016-12-02 - HOUSE ---
Mr. Speaker, I yield 2 minutes to the gentleman from 
Ohio (Mr. Turner), the distinguished chair of the Subcommittee on 
Tactical Air and Land Forces.
...

--- JORDAN (Republican) - 2016-09-14 - HOUSE ---
Mr. Speaker, I yield back the balance of my time.

                          ____________________


</pre></body>
</html>
...

--- GOHMERT (Republican) - 2016-09-12 - HOUSE ---
Mr. Speaker, I move that the House do now adjourn.
  The motion was agreed to; accordingly (at 9 o'clock and 47 minutes 
p.m.), under its previous order, the House adjourned until tomorrow, 
Tuesday, September 13, 2016, at 10 a.m. for morning-hour de
...

--- LOFGREN (Democrat) - 2016-12-07 - HOUSE ---
Mr. Speaker, I had a number of other Members who had 
planned to be here but, because of the hour, there is a conflicting 
event, so they will be adding their voices to the Record.
  Let me just close by saying that our delegation--really, all the 
C
...

--- P

### Interpreting the gap-filling period results

A confirmed run of the two representative weeks above (after the
pagination fix and state-based disambiguation) showed 98.0% matched
for the September week and 99.0% for the November week - both a
substantial improvement over the 81.0% and 89.7% seen before those
changes, and now consistent with the match rates seen elsewhere in
this dataset. This is further evidence that the pagination fix and
state tie-breaker benefited this period in the same way as the rest of
the dataset, not just the years tested directly in
`02_govinfo_dataset.ipynb`, Section 8.